In [ ]:
# UPLOAD RAW DATASET
from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith('.xlsx'):
    df = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file type")


In [ ]:
# Loads Excel file into a DataFrame (like a table)
filename = list(uploaded.keys())[0]

# Auto-detect file type
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith('.xlsx') or filename.endswith('.xls'):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename, sep=None, engine='python')

# Shows the first 5 rows- to check if it loaded correctly.
df.head()

target_col = 'Risk Level'

# Normalize
df[target_col] = df[target_col].astype(str).str.strip().str.lower()

# Map
risk_order = {
    "low": 0,
    "moderate": 1,
    "high": 2
}

df[target_col] = df[target_col].map(risk_order)

# Remove invalid/unmapped rows
df = df[df[target_col].notna()].copy()

# Check
print("Unique after mapping:", df[target_col].unique())

In [ ]:
# LOAD TRAIN-TEST DATA

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob
import os
import joblib

# Locate latest processed dataset folder
folders = glob.glob("/content/drive/MyDrive/data_after_corr_*")

if len(folders) == 0:
    raise ValueError(
        "No saved dataset found."
    )

latest_folder = max(
    folders,
    key=os.path.getmtime
)

print("Loading data from:")
print(latest_folder)

# Load EXACT SAME train-test split
X_train = joblib.load(
    os.path.join(
        latest_folder,
        "X_train_final.pkl"
    )
)

X_test = joblib.load(
    os.path.join(
        latest_folder,
        "X_test_final.pkl"
    )
)

y_train = joblib.load(
    os.path.join(
        latest_folder,
        "y_train.pkl"
    )
)

y_test = joblib.load(
    os.path.join(
        latest_folder,
        "y_test.pkl"
    )
)

print("\nLoaded Successfully")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
FEATURES = [
    'RISK  - Risk Type',
    'Exercise Habit - Frequency',
    'Exercise Habit - Mode',
    'Walking',
    'Diagnosis',
    'Gait',
    'Risk Factor - DM',
    'Total_Muscle_Power',
    'Smoking',
    'Functional Activity',
    'Posture',
    'ROM',
    'Test Today - METS',
    'Marital Status',
    'Risk Factor - ECHO - EF'
]

# CREATE FEATURE-SELECTED DATA

# Check missing features FIRST
missing = [
    f for f in FEATURES
    if f not in X_train.columns
]

print("\nMissing features:")
print(missing)

# STOP if features missing
if len(missing) > 0:

    print("\nAvailable columns:")
    print(X_train.columns.tolist())

    raise ValueError(
        "Some selected features do not exist in X_train."
    )

# USE EXACT ORIGINAL FEATURES
X_train_sel = X_train[FEATURES].copy()
X_test_sel = X_test[FEATURES].copy()

print("\nTrain Shape:", X_train_sel.shape)
print("Test Shape:", X_test_sel.shape)

# FINAL CLEANING

# Fill NaN ONLY
X_train_sel = X_train_sel.fillna(-999)
X_test_sel = X_test_sel.fillna(-999)

print("\nNaN in X_train_sel:",
      X_train_sel.isna().sum().sum())

print("NaN in X_test_sel:",
      X_test_sel.isna().sum().sum())

# EVALUATION FUNCTIONS

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate(y_test, y_pred):

    return {

        "Accuracy":
        accuracy_score(
            y_test,
            y_pred
        ) * 100,

        "Precision (W)":
        precision_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "Recall (W)":
        recall_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "F1 (W)":
        f1_score(
            y_test,
            y_pred,
            average='weighted',
            zero_division=0
        ) * 100,

        "Precision (Macro)":
        precision_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,

        "Recall (Macro)":
        recall_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,

        "F1 (Macro)":
        f1_score(
            y_test,
            y_pred,
            average='macro',
            zero_division=0
        ) * 100,
    }

def print_results(results):

    for k, v in results.items():

        print(f"{k}: {v:.2f}%")

# DECISION TREE MODEL

import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

# SAME CV AS ORIGINAL
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# SAME MODEL AS ORIGINAL
dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

# SAME PARAM GRID
param_grid = {
    "max_depth": list(range(2, 30))
}

# SAME RANDOM SEARCH
total_space = np.prod(
    [len(v) for v in param_grid.values()]
)

n_iter = min(10, total_space)

search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_grid,
    n_iter=n_iter,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

# TRAIN
search.fit(
    X_train_sel,
    y_train
)

print("\nBest Params:")
print(search.best_params_)

best_model = search.best_estimator_

feature_encoders = joblib.load(
    "/content/drive/MyDrive/Best_Models/Encoders/feature_encoders.pkl"
)

# TEST
y_pred = search.predict(X_test_sel)

# RESULTS
print("\nDECISION TREE RESULTS")

print_results(
    evaluate(y_test, y_pred)
)

In [ ]:
# SAVE FINAL DATASET

from google.colab import drive

drive.mount('/content/drive')

output_path = (
    "/content/drive/MyDrive/"
    "predicted_risk_dataset.csv"
)

df.to_csv(
    output_path,
    index=False
)

print("\nSaved to:")
print(output_path)


In [ ]:
# LOAD FEATURE ENCODERS
feature_encoders = joblib.load(
    "/content/drive/MyDrive/Best_Models/Encoders/feature_encoders.pkl"
)

print("Loaded feature encoders:")
print(list(feature_encoders.keys()))



# UPLOAD SYNTHETIC RAW DATASET
from google.colab import files
uploaded = files.upload()

import pandas as pd

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df_raw = pd.read_csv(filename)
elif filename.endswith('.xlsx') or filename.endswith('.xls'):
    df_raw = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file type")

print("\nRaw dataset shape:", df_raw.shape)


In [ ]:
# CREATE DERIVED FEATURE
muscle_cols = [
    'Muscle Power - UL - Right',
    'Muscle Power - UL - Left',
    'Muscle Power - LL - Right',
    'Muscle Power - LL - Left'
]

# Same logic used during preprocessing:
# if a muscle column does not exist, create it as 0

for col in muscle_cols:
    if col not in df_raw.columns:
        df_raw[col] = 0

df_raw['Total_Muscle_Power'] = df_raw[muscle_cols].sum(axis=1)



# SELECT EXACT 15 FEATURES


X_full = df_raw[FEATURES].copy()



# CONVERT FREQUENCY TO NUMERIC


def convert_frequency(x):

    if pd.isna(x):
        return -999

    x = str(x).strip().lower()

    if x == "unknown":
        return -999

    # Example:
    # "7 times per week" -> 7
    # "1-2 times per week" -> 1.5
    # "5-6 times per week" -> 5.5

    import re

    numbers = re.findall(r'\d+(?:\.\d+)?', x)

    if len(numbers) == 0:
        return -999

    numbers = [float(n) for n in numbers]

    if len(numbers) == 1:
        return numbers[0]

    return sum(numbers) / len(numbers)


X_full['Exercise Habit - Frequency'] = (
    X_full['Exercise Habit - Frequency']
    .apply(convert_frequency)
)



# CONVERT ECHO EF


# Training data expects:
# reduced    -> 1
# borderline -> 2
# normal     -> 3

if 'Risk Factor - ECHO - EF' in X_full.columns:

    echo_map = {
        'reduced': 1,
        'borderline': 2,
        'normal': 3
    }

    X_full['Risk Factor - ECHO - EF'] = (
        X_full['Risk Factor - ECHO - EF']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(echo_map)
        .fillna(-999)
    )



# APPLY SAME TRAINING ENCODERS


encoded_columns = []

for col in FEATURES:

    if col in feature_encoders:

        encoder = feature_encoders[col]

        # Create mapping from training encoder
        mapping = {
            str(category): int(code)
            for code, category
            in enumerate(encoder.classes_)
        }

        # Normalize raw values
        values = (
            X_full[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # Apply mapping
        X_full[col] = values.map(mapping)

        # IMPORTANT:
        # unseen categories such as "unknown"
        # become -1 instead of causing an error
        X_full[col] = X_full[col].fillna(-1).astype(int)

        encoded_columns.append(col)

# Make sure every model feature is numeric
non_numeric = X_full.select_dtypes(exclude='number').columns.tolist()

if len(non_numeric) > 0:
    raise ValueError(
        f"These features are still non-numeric: {non_numeric}"
    )


# FINAL NUMERIC CLEANING
X_full = X_full.fillna(-999)


print("\nEncoded columns:")
print(encoded_columns)

print("\nX_full dtypes:")
print(X_full.dtypes)

print("\nNon-numeric columns:")
print(
    X_full.select_dtypes(exclude='number').columns.tolist()
)

print("\nX_full shape:", X_full.shape)



# IMPORTANT ROW CHECK


assert len(X_full) == len(df_raw)

assert X_full.shape[1] == len(FEATURES)

print("\nRows preserved:", len(df_raw))
print("Features:", X_full.shape[1])



# PREDICT ALL RAW PATIENTS


y_pred_full = best_model.predict(X_full).flatten()

print("\nNumber of predictions:", len(y_pred_full))
print("Number of raw rows:", len(df_raw))

assert len(y_pred_full) == len(df_raw)



# CONVERT PREDICTION TO RISK LABEL


reverse_map = {
    0: "low",
    1: "moderate",
    2: "high"
}

predicted_labels = [
    reverse_map[int(i)]
    for i in y_pred_full
]



# REPLACE ORIGINAL RISK LEVEL


df_raw["Risk Level"] = predicted_labels

df_raw["Predicted Risk Level Encoded"] = y_pred_full.astype(int)

# FINAL CHECK
print("\nFINAL DATASET")

print("Original raw rows:", len(df_raw))
print("Final rows:", len(df_raw))

print("\nRisk Level distribution:")
print(df_raw["Risk Level"].value_counts())

assert len(df_raw) == len(predicted_labels)


# SAVE FINAL DATASET

output_path = (
    "/content/drive/MyDrive/"
    "dataset_with_predicted_risk.csv"
)

df_raw.to_csv(
    output_path,
    index=False
)

print("\nSaved to:")
print(output_path)